# 04. Evaluar y visualizar modelos

Este notebook compara las predicciones locales generadas por el notebook 03 contra los `reference_claim` anotados de `test.csv`. METEOR, ROUGE-L y BERTScore miden similitud con la referencia; no validan la veracidad médica de una noticia.

Se ejecuta completo con *Run All* y tiene dos partes:

- **Parte A. Evaluar cada variante.** Recorre las variantes que ya tienen predicciones completas y guarda sus métricas, tablas y figuras dentro de su carpeta de corrida, en `artifacts/03_corridas/`. Las variantes ya evaluadas se reutilizan.
- **Parte B. Resultados finales.** Cuando las cuatro variantes están evaluadas, toma cada modelo con la variante elegida por validación y guarda en `artifacts/05_resultados_finales/` las tablas y figuras de la tesis. Si falta alguna variante, se detiene e indica cuál.

## Requisitos

Ejecuta antes el notebook 03. Para METEOR y BERTScore, instala `pip install -e '.[analysis,models,dev]'` y descarga una vez los recursos de NLTK con `python -m nltk.downloader wordnet omw-1.4`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

DATASET_NAME = 'claims_1500'  # debe coincidir con el notebook 03
FINAL_DIR = ROOT / 'artifacts' / '02_conjuntos' / DATASET_NAME
from claim_extraction.workflow import BASE_SEED, DATASET_SPLIT_SIZES, EXPERIMENT_VARIANTS, run_name_for

RUNS_DIR = ROOT / 'artifacts' / '03_corridas'
REEVALUATE = False  # True recalcula las métricas de variantes ya evaluadas
# Parte B: cada modelo con la variante elegida por validación.
FINAL_RESULTS_DIR = ROOT / 'artifacts' / '05_resultados_finales' / DATASET_NAME
EXPECTED_MODEL_IDS = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']
EXPECTED_SPLIT_SIZES = DATASET_SPLIT_SIZES[DATASET_NAME]

print({'dataset': DATASET_NAME, 'runs_dir': str(RUNS_DIR), 'final_results_dir': str(FINAL_RESULTS_DIR)})

In [ ]:
import json

import pandas as pd

from claim_extraction.workflow import assert_run_matches, evaluate_predictions, split_fingerprints, utc_now, validate_sem1_splits

SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
splits = validate_sem1_splits(SPLIT_PATHS, expected_sizes=EXPECTED_SPLIT_SIZES)
test = splits['test']
SPLIT_FINGERPRINTS = split_fingerprints(splits)


def load_run(variant):
    """Devuelve las predicciones de una variante, o None si el notebook 03 aún no la completa."""
    run_dir = RUNS_DIR / run_name_for(DATASET_NAME, variant)
    manifest_path = run_dir / 'run_manifest.json'
    if not manifest_path.is_file():
        print(f'{variant}: sin corrida todavía; se omite.')
        return None
    run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert_run_matches(run_manifest, dataset_name=DATASET_NAME, variant=variant, fingerprints=SPLIT_FINGERPRINTS, seed=BASE_SEED)
    run_models = run_manifest.get('models', {})
    model_order = [model_id for model_id in EXPECTED_MODEL_IDS if model_id in run_models]
    if not model_order:
        raise ValueError(f'El manifiesto de {run_dir.name} no contiene IDs de modelos reconocidos.')
    prediction_paths = [run_dir / 'predictions' / f'{model_id}.csv' for model_id in model_order]
    missing = [path.name for path in prediction_paths if not path.is_file()]
    if missing:
        print(f"{variant}: faltan predicciones ({', '.join(missing)}); se omite hasta que el notebook 03 las genere.")
        return None
    predictions = pd.concat(
        [pd.read_csv(path, dtype=str, keep_default_na=False) for path in prediction_paths], ignore_index=True
    )
    # Se verifica que cada modelo de esta corrida cubra exactamente las noticias de test.
    expected_test_ids = set(test['record_id'])
    for model_id, group in predictions.groupby('model_id', sort=True):
        if set(group['record_id']) != expected_test_ids or len(group) != len(test):
            raise ValueError(f'{run_dir.name}/{model_id} no cubre exactamente los casos de test.csv.')
    return {'variant': variant, 'run_dir': run_dir, 'run_models': run_models, 'model_order': model_order, 'predictions': predictions}


READY_RUNS = [run for run in map(load_run, EXPERIMENT_VARIANTS) if run is not None]
if not READY_RUNS:
    raise RuntimeError('Ninguna variante tiene predicciones completas. Ejecuta antes el notebook 03.')
print({'test_rows': len(test), 'variantes_listas': [run['variant'] for run in READY_RUNS]})

## Parte A. Evaluar cada variante

Para cada variante, `summary.csv` guarda el promedio de cada modelo y `per_prediction.csv` el puntaje de cada noticia, en la carpeta de su corrida. Junto a ellos quedan la tabla de métricas, los diagnósticos de las salidas (vacías, muy cortas o repetidas), las métricas por tema y las figuras de métricas y de curvas de entrenamiento. Los cuerpos de noticias, claims de referencia y predicciones individuales permanecen ignorados por Git.

Las curvas se leen de los historiales producidos por `Seq2SeqTrainer`. La línea vertical marca el checkpoint seleccionado por menor `eval_loss` dentro de cada arquitectura; no compara pérdidas entre arquitecturas. En la tabla por tema, con grupos pequeños la media es poco estable: revisa siempre `n` antes de comparar temas.

In [ ]:
import matplotlib.pyplot as plt

from claim_extraction.analysis import metrics_by_topic
from claim_extraction.workflow import training_history

METRIC_LABELS = {'meteor': 'METEOR', 'rouge_l_f1': 'ROUGE-L F1', 'bertscore_f1': 'BERTScore F1'}


def metrics_table_from(model_summary):
    return model_summary[['model_id', 'model_display_name', 'prediction_n', 'reference_n', *METRIC_LABELS]].copy()


def diagnostics_table_from(model_summary):
    # Describe las salidas generadas; no mide veracidad médica.
    table = model_summary[[
        'model_display_name', 'prediction_n', 'empty_rate',
        'shorter_than_5_tokens_rate', 'mean_prediction_tokens_operational',
        'duplicate_output_rate',
    ]].copy().rename(columns={
        'model_display_name': 'Modelo',
        'prediction_n': 'Afirmaciones generadas (n)',
        'empty_rate': 'Salidas vacías (%)',
        'shorter_than_5_tokens_rate': 'Menos de 5 tokens (%)',
        'mean_prediction_tokens_operational': 'Longitud media (tokens léxicos)',
        'duplicate_output_rate': 'Repeticiones adicionales (%)',
    })
    for column in ('Salidas vacías (%)', 'Menos de 5 tokens (%)', 'Repeticiones adicionales (%)'):
        table[column] = (100 * table[column]).round(1)
    table['Longitud media (tokens léxicos)'] = table['Longitud media (tokens léxicos)'].round(2)
    return table


def plot_metrics(metrics_table, path, title=None):
    figure, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout='constrained')
    for axis, (column, label) in zip(axes, METRIC_LABELS.items(), strict=True):
        bars = axis.bar(metrics_table['model_display_name'], metrics_table[column], color='#277f8e')
        axis.bar_label(bars, labels=[f'{value:.3f}' for value in metrics_table[column]], padding=3, fontsize=9)
        axis.set(title=label, ylabel='Puntaje medio', ylim=(0, 1))
        axis.tick_params(axis='x', rotation=20)
        axis.grid(axis='y', alpha=.2)
    if title:
        figure.suptitle(title)
    figure.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()


def plot_training_curves(model_dirs, display_names, path, title=None):
    """Dibuja la pérdida de entrenamiento y validación de cada modelo y marca la época elegida."""
    figure, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
    for axis in axes.flat[len(model_dirs):]:
        axis.set_visible(False)  # una corrida puede incluir menos de cuatro modelos
    chosen = []
    for axis, (model_id, model_dir) in zip(axes.flat, model_dirs.items()):
        history = training_history(model_dir).get('log_history', [])
        training = [(row['epoch'], row['loss']) for row in history if 'loss' in row and 'epoch' in row]
        validation = [(row['epoch'], row['eval_loss']) for row in history if 'eval_loss' in row and 'epoch' in row]
        if not training or not validation:
            raise AssertionError(f'Faltan pérdidas de entrenamiento o validación para {model_id}.')
        best_epoch = min(validation, key=lambda point: point[1])[0]
        axis.plot(*zip(*training), label='Entrenamiento', color='#245b78')
        axis.plot(*zip(*validation), label='Validación', marker='o', color='#c77c35')
        axis.axvline(best_epoch, color='#777777', linestyle=':', label=f'Checkpoint elegido (época {best_epoch:g})')
        axis.set(title=display_names[model_id], xlabel='Época', ylabel='Pérdida')
        axis.grid(alpha=.2)
        axis.legend(fontsize=8, frameon=False)
        chosen.append({'model_id': model_id, 'checkpoint_epoch': best_epoch})
    if title:
        figure.suptitle(title)
    figure.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    return pd.DataFrame(chosen)

### Calcular métricas, tablas y figuras de cada variante

Las métricas de una variante ya evaluada se reutilizan; cambia `REEVALUATE` a `True` para recalcularlas. Si el notebook 03 agregó modelos a una variante, esa variante se vuelve a evaluar.

In [ ]:
# Estas métricas comparan texto generado y referencia; no determinan veracidad médica.
for run in READY_RUNS:
    variant, RUN_DIR, model_order = run['variant'], run['run_dir'], run['model_order']
    EVALUATION_DIR = RUN_DIR / 'evaluation'
    TABLES_DIR = EVALUATION_DIR / 'tables'
    FIGURES_DIR = EVALUATION_DIR / 'figures'
    summary_path = EVALUATION_DIR / 'summary.csv'
    detail_path = EVALUATION_DIR / 'per_prediction.csv'
    print(f'=== Variante {variant} ===')

    evaluated = (
        summary_path.is_file() and detail_path.is_file()
        and set(model_order) <= set(pd.read_csv(summary_path, encoding='utf-8')['model_id'])
    )
    if evaluated and not REEVALUATE:
        summary = pd.read_csv(summary_path, encoding='utf-8')
        per_prediction = pd.read_csv(detail_path, dtype={'record_id': str}, keep_default_na=False, encoding='utf-8')
        print('Métricas ya calculadas; se reutilizan.')
    else:
        summary, per_prediction = evaluate_predictions(run['predictions'], test)
        if summary[['meteor', 'rouge_l_f1', 'bertscore_f1']].isna().any().any():
            raise AssertionError(f'{variant}: no se calcularon las tres métricas de referencia para todos los modelos.')
        EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
        summary.to_csv(summary_path, index=False, encoding='utf-8')
        per_prediction.to_csv(detail_path, index=False, encoding='utf-8')
        status_path = RUN_DIR / 'run_status.json'
        status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
        status['status'] = 'completed'
        status['evaluated_at_utc'] = utc_now()
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
    summary = summary.set_index('model_id').reindex(model_order).reset_index()

    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    metrics_table = metrics_table_from(summary)
    metrics_table.to_csv(TABLES_DIR / '01_metricas_principales.csv', index=False, encoding='utf-8')
    if 'tema' in test:
        metrics_by_topic(per_prediction, test).to_csv(TABLES_DIR / '02_metricas_por_tema.csv', index=False, encoding='utf-8')
    else:
        print('Este conjunto no tiene tema; se omite la tabla por tema.')
    diagnostics_table_from(summary).to_csv(TABLES_DIR / '03_diagnosticos_generacion.csv', index=False, encoding='utf-8')
    plot_metrics(metrics_table, FIGURES_DIR / '01_comparacion_metricas.png', title=f'Variante {variant}')
    plot_training_curves(
        {model_id: RUN_DIR / 'models' / model_id for model_id in model_order},
        {model_id: run['run_models'][model_id]['display_name'] for model_id in model_order},
        FIGURES_DIR / '02_curvas_entrenamiento_validacion.png',
        title=f'Variante {variant}',
    )
    display(metrics_table)
    print(f'Resultados de la variante: {EVALUATION_DIR.relative_to(ROOT)}')

### Comparar variantes

Reúne las variantes evaluadas. La columna `elegida_por_validacion` marca, para cada modelo, la variante con menor pérdida de validación. Esa elección no usa el test: METEOR, ROUGE-L y BERTScore solo se reportan.

In [ ]:
# Reúne las variantes que la parte A ya evaluó.
comparison_rows = []
for variant in EXPERIMENT_VARIANTS:
    variant_dir = RUNS_DIR / run_name_for(DATASET_NAME, variant)
    summary_path = variant_dir / 'evaluation' / 'summary.csv'
    if not summary_path.is_file():
        continue
    variant_summary = pd.read_csv(summary_path, encoding='utf-8').set_index('model_id')
    for model_id in EXPECTED_MODEL_IDS:
        if model_id not in variant_summary.index:
            continue
        evaluations = [row for row in training_history(variant_dir / 'models' / model_id)['log_history'] if 'eval_loss' in row]
        best = min(evaluations, key=lambda row: row['eval_loss'])
        metrics = variant_summary.loc[model_id]
        comparison_rows.append({
            'modelo': metrics['model_display_name'], 'model_id': model_id, 'variante': variant,
            'epocas_ejecutadas': round(evaluations[-1]['epoch']), 'mejor_epoca': round(best['epoch']),
            'perdida_validacion': best['eval_loss'], 'meteor_test': metrics['meteor'],
            'rouge_l_f1_test': metrics['rouge_l_f1'], 'bertscore_f1_test': metrics['bertscore_f1'],
        })
if not comparison_rows:
    raise RuntimeError('Todavía no hay variantes evaluadas.')

comparison = pd.DataFrame(comparison_rows)
comparison['elegida_por_validacion'] = comparison.index.isin(comparison.groupby('model_id')['perdida_validacion'].idxmin())
comparison.to_csv(RUNS_DIR / f'comparacion_variantes_{DATASET_NAME}.csv', index=False, encoding='utf-8')
comparison

## Parte B. Resultados finales

Corre sola cuando las cuatro variantes de los cuatro modelos están entrenadas y evaluadas; si falta alguna, el notebook se detiene aquí y lo indica. Cada modelo entra con la variante que eligió la validación, y todo se guarda en `artifacts/05_resultados_finales/`: es la carpeta que se revisa para redactar la tesis.

La elección de variantes sigue la sección 4.2 de `docs/EXPERIMENT_PROTOCOL.md`. Estos puntajes comparan similitud con la referencia; no validan la veracidad médica.

In [ ]:
from claim_extraction.workflow import load_selected_evaluations

missing_variants = {
    model_id: sorted(set(EXPERIMENT_VARIANTS) - set(comparison.loc[comparison['model_id'].eq(model_id), 'variante']))
    for model_id in EXPECTED_MODEL_IDS
}
missing_variants = {model_id: variants for model_id, variants in missing_variants.items() if variants}
if missing_variants:
    raise RuntimeError(f'Faltan variantes por evaluar antes de los resultados finales: {missing_variants}')

selected = (
    comparison[comparison['elegida_por_validacion']]
    .drop(columns='elegida_por_validacion')
    .set_index('model_id').reindex(EXPECTED_MODEL_IDS).reset_index()
)
SELECTED_VARIANTS = dict(zip(selected['model_id'], selected['variante'], strict=True))
final_summary, final_per_prediction = load_selected_evaluations(
    RUNS_DIR,
    dataset_name=DATASET_NAME,
    selected_variants=SELECTED_VARIANTS,
    fingerprints=split_fingerprints(splits),
)
final_summary = final_summary.set_index('model_id').reindex(EXPECTED_MODEL_IDS).reset_index()
final_names = dict(zip(final_summary['model_id'], final_summary['model_display_name'], strict=True))

FINAL_TABLES_DIR = FINAL_RESULTS_DIR / 'tablas'
FINAL_FIGURES_DIR = FINAL_RESULTS_DIR / 'figuras'
FINAL_TABLES_DIR.mkdir(parents=True, exist_ok=True)
FINAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

split_table = pd.DataFrame([{'split': name, 'filas': len(frame)} for name, frame in splits.items()])
final_metrics = metrics_table_from(final_summary)
final_metrics.insert(2, 'variante', final_summary['variante'].to_numpy())
final_diagnostics = diagnostics_table_from(final_summary)
final_diagnostics.insert(1, 'Variante', final_summary['variante'].to_numpy())

split_table.to_csv(FINAL_TABLES_DIR / '01_distribucion_particiones.csv', index=False, encoding='utf-8')
selected.to_csv(FINAL_TABLES_DIR / '02_variantes_elegidas.csv', index=False, encoding='utf-8')
final_metrics.to_csv(FINAL_TABLES_DIR / '03_metricas_finales.csv', index=False, encoding='utf-8')
final_diagnostics.to_csv(FINAL_TABLES_DIR / '05_diagnosticos_generacion.csv', index=False, encoding='utf-8')

plot_metrics(final_metrics, FINAL_FIGURES_DIR / '01_comparacion_metricas.png')
plot_training_curves(
    {model_id: RUNS_DIR / run_name_for(DATASET_NAME, variant) / 'models' / model_id for model_id, variant in SELECTED_VARIANTS.items()},
    final_names,
    FINAL_FIGURES_DIR / '02_curvas_entrenamiento_validacion.png',
)
display(selected)
final_metrics

### Resultados por tema

Las mismas métricas de la tabla final, separadas por el `tema` anotado. Revisa `n` antes de comparar temas.

In [ ]:
if 'tema' in test:
    final_by_topic = metrics_by_topic(final_per_prediction, test)
    final_by_topic.to_csv(FINAL_TABLES_DIR / '04_metricas_por_tema.csv', index=False, encoding='utf-8')
    display(final_by_topic)
else:
    print('Este conjunto no tiene tema; se omite la tabla por tema.')

### Distribución de longitudes

Describe el conjunto, no una variante, por eso se calcula una sola vez. La longitud de noticia combina `title` y `body`; la de referencia corresponde al `reference_claim` anotado.

In [ ]:
# Esta figura describe el tamaño del material de entrada y de las afirmaciones, no la calidad del modelo.
import re

all_records = pd.concat([splits[name] for name in ('train', 'validation', 'test')], ignore_index=True)
word_count = lambda value: len(re.findall(r'\b\w+\b', str(value), flags=re.UNICODE))
article_text = all_records['title'].fillna('') + ' ' + all_records['body'].fillna('')
lengths = pd.DataFrame({
    'news_words_title_body': article_text.map(word_count),
    'reference_claim_words': all_records['reference_claim'].fillna('').map(word_count),
})
if len(lengths) != sum(EXPECTED_SPLIT_SIZES.values()):
    raise AssertionError('La distribución debe incluir exactamente los registros del conjunto final.')

figure, axes = plt.subplots(2, 1, figsize=(8, 7), layout='constrained')
for axis, column, title, color in [
    (axes[0], 'news_words_title_body', 'Noticias (título y cuerpo)', '#245b78'),
    (axes[1], 'reference_claim_words', 'Afirmaciones de referencia', '#27877c'),
]:
    median = lengths[column].median()
    axis.hist(lengths[column], bins=20, color=color, edgecolor='white')
    axis.axvline(median, color='#b75a3c', linestyle='--', label=f'Mediana: {median:.0f}')
    axis.set(title=title, xlabel='Longitud (palabras)', ylabel='Frecuencia')
    axis.legend(frameon=False)
    axis.grid(axis='y', alpha=.2)
figure.savefig(FINAL_FIGURES_DIR / '03_distribucion_longitudes.png', dpi=180, bbox_inches='tight')
plt.show()

lengths.describe().T[['count', 'mean', '50%', 'min', 'max']]

### Cantidad de tokens

Cada modelo divide el texto con su propio tokenizador, así que el mismo texto ocupa una cantidad distinta de tokens en cada uno. Las figuras cuentan los tokens sin truncar sobre todos los registros del conjunto final y marcan el límite usado en el entrenamiento: lo que lo supera no llega al modelo.

In [ ]:
# Se cuenta sin truncar para ver cuánto texto queda fuera del límite de entrenamiento.
from claim_extraction.models import load_tokenizer
from claim_extraction.registry import get_model
from claim_extraction.text import input_for_model

input_texts = [input_for_model(row) for row in all_records.to_dict(orient='records')]
target_texts = all_records['reference_claim'].map(str).tolist()

token_rows = []
token_limits = {}
for model_id in EXPECTED_MODEL_IDS:
    spec = get_model(model_id)
    tokenizer = load_tokenizer(spec.base_checkpoint, model_id=model_id)
    token_limits[model_id] = {
        'source_tokens': int(spec.training['max_source_length']),
        'target_tokens': int(spec.training['max_target_length']),
    }
    source_ids = tokenizer(input_texts, truncation=False, verbose=False)['input_ids']
    target_ids = tokenizer(text_target=target_texts, truncation=False, verbose=False)['input_ids']
    for source, target in zip(source_ids, target_ids, strict=True):
        token_rows.append({'model_id': model_id, 'source_tokens': len(source), 'target_tokens': len(target)})
token_counts = pd.DataFrame(token_rows)


def plot_token_histograms(column, title, color, filename):
    figure, axes = plt.subplots(2, 2, figsize=(12, 7), layout='constrained', sharex=True, sharey=True)
    for axis, model_id in zip(axes.flat, EXPECTED_MODEL_IDS, strict=True):
        counts = token_counts.loc[token_counts['model_id'].eq(model_id), column]
        limit = token_limits[model_id][column]
        axis.hist(counts, bins=30, color=color, edgecolor='white')
        axis.axvline(counts.median(), color='#777777', linestyle=':', label=f'Mediana: {counts.median():.0f}')
        axis.axvline(limit, color='#b75a3c', linestyle='--', label=f'Límite: {limit} ({counts.gt(limit).mean():.1%} lo supera)')
        axis.set(title=final_names[model_id], xlabel='Tokens', ylabel='Noticias')
        axis.grid(axis='y', alpha=.2)
        axis.legend(fontsize=9, frameon=False)
    figure.suptitle(title)
    figure.savefig(FINAL_FIGURES_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()


plot_token_histograms('source_tokens', 'Tokens de entrada (título y cuerpo)', '#245b78', '04_tokens_noticias.png')
plot_token_histograms('target_tokens', 'Tokens de las afirmaciones de referencia', '#27877c', '05_tokens_afirmaciones.png')

token_summary = pd.DataFrame([
    {
        'Modelo': final_names[model_id],
        'Texto': label,
        'Mediana': group[column].median(),
        'Percentil 95': group[column].quantile(.95),
        'Máximo': group[column].max(),
        'Límite': token_limits[model_id][column],
        'Superan el límite (%)': round(100 * group[column].gt(token_limits[model_id][column]).mean(), 1),
    }
    for model_id, group in token_counts.groupby('model_id', sort=False)
    for column, label in [('source_tokens', 'Noticia'), ('target_tokens', 'Afirmación')]
])
token_summary.to_csv(FINAL_TABLES_DIR / '06_resumen_tokens.csv', index=False, encoding='utf-8')
print(f'Resultados finales: {FINAL_RESULTS_DIR.relative_to(ROOT)}')
token_summary